In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
from dataclasses import dataclass
from option_env import OptionPricingEnv

np.random.seed(42)

@dataclass
class Params:
    S0: float = 100.0
    K: float = 100.0
    T: float = 1.0
    r: float = 0.05
    sigma: float = 0.5
    N: int = 300    # number of exercise dates
    M: int = 1 # number of processes (1 = standard, >1 = Asian-style)
    L: int =  3  # exercise rights
    delta: int = 0    # refractory period between exercises (0 = next step allowed)
    
    @property
    def dt(self): return self.T / self.N
    
    @property
    def df(self): return np.exp(-self.r * self.dt)

params = Params()
print(f"American Put: S0={params.S0}, K={params.K}, T={params.T}, r={params.r}, σ={params.sigma}, N={params.N}, M={params.M}, L={params.L}, delta={params.delta}")
print(f"dt={params.dt:.4f}, discount_factor={params.df:.6f}")

American Put: S0=100.0, K=100.0, T=1.0, r=0.05, σ=0.5, N=300, M=1, L=3, delta=0
dt=0.0033, discount_factor=0.999833


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


### Path Geberation

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# PATH GENERATION  — unified interface for all process types
# ══════════════════════════════════════════════════════════════════════════════
#
# generate_paths(params, n_paths, process='gbm', seed=None, randomize_S0=True)
#
# process options
# ───────────────
# 'gbm'      Geometric Brownian Motion  (continuous, log-normal increments)
#            The canonical Black-Scholes model.  Used by compute_exact_boundary_continuous.
#
# 'binomial' CRR Binomial Tree  (discrete price, discrete time)
#            At each step: S → S·u with prob p_u, S → S·d with prob p_d.
#            u = exp(σ√dt), d = 1/u, p_u = (exp(r·dt)-d)/(u-d).
#
# 'trinomial' Boyle Trinomial Tree (discrete price, discrete time)
#            At each step: S → S·u, S → S·m (flat), or S → S·d.
#            Solves the "sawtooth" boundary issue of the binomial model.
#
# 'iid_gaussian'
#            Each log-return is drawn i.i.d. N(μ·dt, σ²·dt).
#
# All variants return shape (n_paths, N+1, M) and support randomize_S0.
# M>1 is only meaningful for 'gbm' and 'iid_gaussian'.

import numpy as np

def _crr_params(params):
    """CRR up/down factors and risk-neutral probabilities for Binomial Tree."""
    dt = params.dt
    u = np.exp(params.sigma * np.sqrt(dt))
    d = 1.0 / u
    p_u = (np.exp(params.r * dt) - d) / (u - d)
    assert 0 < p_u < 1, f"p_u={p_u:.4f} outside (0,1) — reduce sigma or increase N"
    return u, d, p_u, 1.0 - p_u

def _trinomial_params(params):
    """Boyle (1986) parameters for Trinomial Tree."""
    dt = params.dt
    # lambda is typically sqrt(3) to ensure probabilities stay positive and stable
    lam = np.sqrt(3.0) 
    
    u = np.exp(lam * params.sigma * np.sqrt(dt))
    d = 1.0 / u
    m = 1.0
    
    nu = params.r - 0.5 * params.sigma**2
    
    p_u = 1.0 / (2.0 * lam**2) + (nu * np.sqrt(dt)) / (2.0 * lam * params.sigma)
    p_d = 1.0 / (2.0 * lam**2) - (nu * np.sqrt(dt)) / (2.0 * lam * params.sigma)
    p_m = 1.0 - p_u - p_d
    
    assert 0 < p_u < 1 and 0 < p_d < 1 and 0 < p_m < 1, "Trinomial probs outside (0,1). Check parameters."
    
    return u, m, d, p_u, p_m, p_d

def generate_paths(params, n_paths, process='gbm', seed=None, randomize_S0=True):
    """
    Generate Monte Carlo paths.

    Parameters
    ----------
    params       : Params dataclass
    n_paths      : int
    process      : 'gbm' | 'binomial' | 'trinomial' | 'iid_gaussian'
    seed         : int or None
    randomize_S0 : bool  — scatter S0 in [0.4K, 1.6K] for global policy learning

    Returns
    -------
    paths : np.ndarray, shape (n_paths, N+1, M)
    """
    rng = np.random.default_rng(seed)
    N, M, K = params.N, params.M, params.K

    # Starting prices
    if randomize_S0:
        S0 = rng.uniform(K * 0.4, K * 1.6, (n_paths, M))
    else:
        S0 = np.full((n_paths, M), params.S0)

    paths = np.zeros((n_paths, N + 1, M))
    paths[:, 0, :] = S0

    if process == 'gbm':
        dt = params.dt
        drift     = (params.r - 0.5 * params.sigma**2) * dt
        diffusion = params.sigma * np.sqrt(dt)
        Z = rng.standard_normal((n_paths, N, M))
        log_ret = np.cumsum(drift + diffusion * Z, axis=1)   # (n, N, M)
        paths[:, 1:, :] = S0[:, np.newaxis, :] * np.exp(log_ret)

    elif process == 'binomial':
        if M != 1:
            raise ValueError("binomial process only supports M=1")
        u, d, p_u, p_d = _crr_params(params)
        # Draw Bernoulli up/down steps
        ups = rng.random((n_paths, N)) < p_u      # True = up move
        for t in range(N):
            paths[:, t+1, 0] = np.where(ups[:, t],
                                         paths[:, t, 0] * u,
                                         paths[:, t, 0] * d)
                                         
    elif process == 'trinomial':
        if M != 1:
            raise ValueError("trinomial process only supports M=1")
        u, m, d, p_u, p_m, p_d = _trinomial_params(params)
        
        # Fast vectorized generation: draw step multipliers and cumulative product
        step_multipliers = rng.choice(
            a=[u, m, d], 
            p=[p_u, p_m, p_d], 
            size=(n_paths, N)
        )
        paths[:, 1:, 0] = S0[:, 0, np.newaxis] * np.cumprod(step_multipliers, axis=1)

    elif process == 'iid_gaussian':
        # Each step i.i.d.  — NOT cumulative, so no autocorrelation across steps
        dt = params.dt
        drift     = (params.r - 0.5 * params.sigma**2) * dt
        diffusion = params.sigma * np.sqrt(dt)
        Z = rng.standard_normal((n_paths, N, M))
        log_steps = drift + diffusion * Z
        paths[:, 1:, :] = S0[:, np.newaxis, :] * np.exp(np.cumsum(log_steps, axis=1))

    else:
        raise ValueError(f"Unknown process '{process}'. Choose gbm | binomial | trinomial | iid_gaussian")

    return paths


# ── Quick smoke-tests ─────────────────────────────────────────────────────────
_p = params   # uses the Params() defined in cell 0
train_paths = generate_paths(_p, 50,  process='gbm',       seed=42)
test_paths  = generate_paths(_p, 100, process='gbm',       seed=997)
_bin        = generate_paths(_p, 10,  process='binomial',  seed=0)
_tri        = generate_paths(_p, 10,  process='trinomial', seed=0)
_iid        = generate_paths(_p, 10,  process='iid_gaussian', seed=0)

print(f"GBM train: {train_paths.shape},  test: {test_paths.shape}")
print(f"Binomial sample:      {_bin.shape},  prices[0,0..5,0] = {_bin[0,:6,0].round(2)}")
print(f"Trinomial sample:     {_tri.shape},  prices[0,0..5,0] = {_tri[0,:6,0].round(2)}")
print(f"IID-Gaussian sample:  {_iid.shape},  prices[0,0..5,0] = {_iid[0,:6,0].round(2)}")

GBM train: (50, 301, 1),  test: (100, 301, 1)
Binomial sample:      (10, 301, 1),  prices[0,0..5,0] = [116.44 113.12 116.44 113.12 116.44 113.12]
Trinomial sample:     (10, 301, 1),  prices[0,0..5,0] = [116.44 116.44 122.41 116.44 122.41 122.41]
IID-Gaussian sample:  (10, 301, 1),  prices[0,0..5,0] = [116.44 114.33 114.44 106.98 106.28 102.5 ]


### Helpers

In [3]:
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor

class BaseAgent:
    def act(self, obs: np.ndarray) -> int:
        """Returns 0 (hold) or 1 (exercise) based on the observation."""
        raise NotImplementedError

    def train(self, env, n_episodes: int):
        """Trains the agent on the environment."""
        raise NotImplementedError


def collect_random_transitions(env, n_episodes):
    """
    Collects a dataset of transitions (s, a, r, s', done) using a random exploratory policy.
    Crucial for offline RL algorithms like FQI and LSPI.
    """
    transitions = []
    for i in range(n_episodes):
        obs, _ = env.reset_to_path(i % env.n_paths)
        done = False
        while not done:
            # We need to explore both holding and exercising
            # Only allow exercise (1) if rights > 0 and not in cooldown
            can_exercise = obs[-1] > 0 and env._cooldown == 0
            valid_actions = [0, 1] if can_exercise else [0]
            
            action = np.random.choice(valid_actions)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            transitions.append((obs, action, reward, next_obs, done))
            obs = next_obs
            
    return transitions

### LSM

In [4]:
import numpy as np
from itertools import combinations_with_replacement

def get_basis_features(S, degree=2):
    """
    Generates polynomial basis features for M assets.
    S: array of shape (n_paths, M)
    Returns: array of shape (n_paths, n_features)
    """
    n_paths, M = S.shape
    features = [np.ones((n_paths, 1))]
    
    # Add combinations of assets up to 'degree'
    for d in range(1, degree + 1):
        for combo in combinations_with_replacement(range(M), d):
            term = np.ones(n_paths)
            for idx in combo:
                term *= S[:, idx]
            features.append(term.reshape(-1, 1))
            
    return np.hstack(features)

def train_lsmc(paths, params, payoff_fn, degree=2, grid_points=1000, is_put=True):
    """
    Trains the LSMC algorithm and extracts the continuous mathematical boundary on the fly.
    """
    n_paths, N_steps, M = paths.shape
    N = N_steps - 1
    L = params.L
    delta = params.delta
    df = params.df
    K = params.K
    
    if M > 1:
         raise ValueError("Continuous boundary extraction is only supported for M=1.")
    
    policy_weights = {t: {l: {} for l in range(1, L + 1)} for t in range(1, N + 1)}
    
    # ─── PRE-COMPUTE EVALUATION GRID ─────────────────────────────────────────────
    # Doing this outside the time loop is where the massive efficiency gain lives
    spot_grid = np.linspace(1e-4, K * 1.2, grid_points).reshape(-1, 1)
    basis_grid = get_basis_features(spot_grid, degree)
    payoff_grid = payoff_fn(spot_grid).flatten()
    
    boundaries_curve = {l: np.full(N + 1, np.nan) for l in range(1, L + 1)}
    
    # Value matrix V[t, l]
    V = np.zeros((N + 1, L + 1, n_paths))
    
    # ─── EXPIRATION (t=N) ────────────────────────────────────────────────────────
    payoff_N = payoff_fn(paths[:, N, :])
    for l in range(1, L + 1):
        V[N, l, :] = np.maximum(0, payoff_N)
        boundaries_curve[l][N] = K if is_put else spot_grid[payoff_grid > 0].min()
            
    # ─── BACKWARD INDUCTION ──────────────────────────────────────────────────────
    for t in range(N - 1, 0, -1):
        S_t = paths[:, t, :]
        X_t = get_basis_features(S_t, degree)
        payoff_t = payoff_fn(S_t)
        itm_mask = payoff_t > 0
        
        for l in range(1, L + 1):
            # 1. Regressions (Fitting)
            Y_cont = V[t + 1, l, :] * df
            beta_cont = np.zeros(X_t.shape[1])
            if np.sum(itm_mask) > 0:
                beta_cont = np.linalg.lstsq(X_t[itm_mask], Y_cont[itm_mask], rcond=None)[0]
            C_t = X_t @ beta_cont
            
            tau = min(t + 1 + delta, N)
            Y_ex = V[tau, l - 1, :] * (df ** (tau - t)) if l > 1 else np.zeros(n_paths)
            beta_ex = np.zeros(X_t.shape[1])
            if l > 1 and np.sum(itm_mask) > 0:
                beta_ex = np.linalg.lstsq(X_t[itm_mask], Y_ex[itm_mask], rcond=None)[0]
                
            E_future = X_t @ beta_ex if l > 1 else np.zeros(n_paths)
            Total_E_t = payoff_t + E_future
            
            # Store weights for forward pass
            policy_weights[t][l] = {'cont': beta_cont, 'ex': beta_ex}
            
            # Update Value Matrix
            exercise_decision = (itm_mask) & (Total_E_t > C_t)
            V[t, l, :] = np.where(exercise_decision, payoff_t + Y_ex, Y_cont)
            
            # 2. On-The-Fly Boundary Extraction (Evaluation)
            cont_grid = basis_grid @ beta_cont
            E_future_grid = basis_grid @ beta_ex if l > 1 else np.zeros(grid_points)
            exercise_val_grid = payoff_grid + E_future_grid
            
            exercise_zone = (exercise_val_grid > cont_grid) & (payoff_grid > 0)
            if np.any(exercise_zone):
                boundaries_curve[l][t] = spot_grid[exercise_zone].max() if is_put else spot_grid[exercise_zone].min()

    return policy_weights, boundaries_curve

### TD-LSM

In [5]:
import numpy as np

def train_tdlsm(paths, params, payoff_fn, degree=2, grid_points=1000, is_put=True):
    """
    Trains the TD-LSM algorithm using bootstrapped continuation values (TD(0)) for ITM paths,
    and realized returns for OTM paths. Extracts continuous boundary on the fly.
    """
    n_paths, N_steps, M = paths.shape
    N = N_steps - 1
    L = params.L
    delta = params.delta
    df = params.df
    K = params.K
    
    if M > 1:
         raise ValueError("Continuous boundary extraction is only supported for M=1.")
    
    policy_weights = {t: {l: {} for l in range(1, L + 1)} for t in range(1, N + 1)}
    
    # Pre-compute evaluation grid for on-the-fly boundary extraction
    spot_grid = np.linspace(1e-4, K * 1.2, grid_points).reshape(-1, 1)
    basis_grid = get_basis_features(spot_grid, degree)
    payoff_grid = payoff_fn(spot_grid).flatten()
    boundaries_curve = {l: np.full(N + 1, np.nan) for l in range(1, L + 1)}
    
    # We still must track the standard realized value matrix for the OTM fallback
    V_realized = np.zeros((N + 1, L + 1, n_paths))
    
    # ─── EXPIRATION (t=N) ────────────────────────────────────────────────────────
    payoff_N = payoff_fn(paths[:, N, :])
    for l in range(1, L + 1):
        V_realized[N, l, :] = np.maximum(0, payoff_N)
        boundaries_curve[l][N] = K if is_put else spot_grid[payoff_grid > 0].min()
            
    # ─── BACKWARD INDUCTION WITH TD BOOTSTRAPPING ────────────────────────────────
    for t in range(N - 1, 0, -1):
        S_t = paths[:, t, :]
        X_t = get_basis_features(S_t, degree)
        payoff_t = payoff_fn(S_t)
        itm_mask_t = payoff_t > 0
        
        # State at t+1 is needed to evaluate the TD target
        S_t1 = paths[:, t + 1, :]
        payoff_t1 = payoff_fn(S_t1)
        itm_mask_t1 = payoff_t1 > 0
        
        for l in range(1, L + 1):
            
            # ═════════════════════════════════════════════════════════════════════
            # 1. CONSTRUCT TD TARGET FOR CONTINUATION (Y_cont)
            # ═════════════════════════════════════════════════════════════════════
            Y_cont_target = np.zeros(n_paths)
            
            # Fallback: OTM paths at t+1 use the realized future cash flows
            Y_cont_target[~itm_mask_t1] = V_realized[t + 1, l, ~itm_mask_t1]
            
            # Bootstrap: ITM paths at t+1 use the t+1 regressor
            if t == N - 1:
                # At expiration, value is purely the immediate payoff
                Y_cont_target[itm_mask_t1] = payoff_t1[itm_mask_t1]
            else:
                X_t1 = get_basis_features(S_t1, degree)
                w_cont_t1 = policy_weights[t + 1][l]['cont']
                C_t1_est = X_t1[itm_mask_t1] @ w_cont_t1
                
                # If L > 1, add the value of future rights at t+1
                E_future_t1 = (X_t1[itm_mask_t1] @ policy_weights[t + 1][l]['ex']) if l > 1 else 0
                
                exercise_val_t1 = payoff_t1[itm_mask_t1] + E_future_t1
                Y_cont_target[itm_mask_t1] = np.maximum(exercise_val_t1, C_t1_est)
                
            Y_cont_td = Y_cont_target * df
            
            # Regress TD target on current state (only for ITM paths)
            beta_cont = np.zeros(X_t.shape[1])
            if np.sum(itm_mask_t) > 0:
                beta_cont = np.linalg.lstsq(X_t[itm_mask_t], Y_cont_td[itm_mask_t], rcond=None)[0]
            
            C_t = X_t @ beta_cont

            # ═════════════════════════════════════════════════════════════════════
            # 2. CONSTRUCT TD TARGET FOR FUTURE RIGHTS (Y_ex) - If Swing Option
            # ═════════════════════════════════════════════════════════════════════
            tau = min(t + 1 + delta, N)
            beta_ex = np.zeros(X_t.shape[1])
            
            if l > 1:
                S_tau = paths[:, tau, :]
                payoff_tau = payoff_fn(S_tau)
                itm_mask_tau = payoff_tau > 0
                
                Y_ex_target = np.zeros(n_paths)
                Y_ex_target[~itm_mask_tau] = V_realized[tau, l - 1, ~itm_mask_tau]
                
                if tau == N:
                    Y_ex_target[itm_mask_tau] = payoff_tau[itm_mask_tau]
                else:
                    X_tau = get_basis_features(S_tau, degree)
                    w_cont_tau = policy_weights[tau][l - 1]['cont']
                    C_tau_est = X_tau[itm_mask_tau] @ w_cont_tau
                    
                    E_future_tau = (X_tau[itm_mask_tau] @ policy_weights[tau][l - 1]['ex']) if (l - 1 > 1) else 0
                    
                    exercise_val_tau = payoff_tau[itm_mask_tau] + E_future_tau
                    Y_ex_target[itm_mask_tau] = np.maximum(exercise_val_tau, C_tau_est)
                    
                Y_ex_td = Y_ex_target * (df ** (tau - t))
                
                if np.sum(itm_mask_t) > 0:
                    beta_ex = np.linalg.lstsq(X_t[itm_mask_t], Y_ex_td[itm_mask_t], rcond=None)[0]

            E_future = X_t @ beta_ex if l > 1 else np.zeros(n_paths)
            Total_E_t = payoff_t + E_future
            
            policy_weights[t][l] = {'cont': beta_cont, 'ex': beta_ex}
            
            # ═════════════════════════════════════════════════════════════════════
            # 3. UPDATE REALIZED V MATRIX (Required for earlier OTM fallbacks)
            # ═════════════════════════════════════════════════════════════════════
            exercise_decision = (itm_mask_t) & (Total_E_t > C_t)
            
            # V_realized MUST strictly track realized Monte Carlo cash flows, not TD estimates
            Y_cont_realized = V_realized[t + 1, l, :] * df
            Y_ex_realized = V_realized[tau, l - 1, :] * (df ** (tau - t)) if l > 1 else np.zeros(n_paths)
            
            V_realized[t, l, :] = np.where(exercise_decision, payoff_t + Y_ex_realized, Y_cont_realized)
            
            # ═════════════════════════════════════════════════════════════════════
            # 4. EXTRACT BOUNDARY (Grid Search)
            # ═════════════════════════════════════════════════════════════════════
            cont_grid = basis_grid @ beta_cont
            E_future_grid = basis_grid @ beta_ex if l > 1 else np.zeros(grid_points)
            exercise_val_grid = payoff_grid + E_future_grid
            
            exercise_zone = (exercise_val_grid > cont_grid) & (payoff_grid > 0)
            if np.any(exercise_zone):
                boundaries_curve[l][t] = spot_grid[exercise_zone].max() if is_put else spot_grid[exercise_zone].min()

    return policy_weights, boundaries_curve

### FQI

In [6]:
class FQIAgent(BaseAgent):
    def __init__(self, n_iterations=10, df=0.95):
        self.n_iterations = n_iterations
        self.df = df
        self.q_model = None # Maps (obs, action) -> Q-value

    def _prepare_features(self, obs, action):
        """Concatenates observation and action into a single feature vector."""
        if obs.ndim == 1:
            return np.append(obs, action)
        return np.column_stack([obs, action])

    def train(self, env, n_episodes):
        transitions = collect_random_transitions(env, n_episodes)
        
        S = np.array([t[0] for t in transitions])
        A = np.array([t[1] for t in transitions])
        R = np.array([t[2] for t in transitions])
        S_next = np.array([t[3] for t in transitions])
        Dones = np.array([t[4] for t in transitions])
        
        X = self._prepare_features(S, A)
        
        # Iteration 0: Q(s,a) initialized to 0, so target is just the reward
        Y = R.copy()
        
        for i in range(self.n_iterations):
            self.q_model = ExtraTreesRegressor(n_estimators=50, min_samples_leaf=5, n_jobs=-1)
            self.q_model.fit(X, Y)
            
            # Predict max Q for the next state: max(Q(s', 0), Q(s', 1))
            X_next_0 = self._prepare_features(S_next, np.zeros_like(A))
            X_next_1 = self._prepare_features(S_next, np.ones_like(A))
            
            Q_next_0 = self.q_model.predict(X_next_0)
            Q_next_1 = self.q_model.predict(X_next_1)
            
            # If rights == 0 in next state, action 1 is invalid. Force its Q-value down.
            rights_next = S_next[:, -1]
            Q_next_1[rights_next == 0] = -np.inf 
            
            V_next = np.maximum(Q_next_0, Q_next_1)
            
            # Update targets for the next iteration
            Y = R + self.df * V_next * (1 - Dones)
            print(f"FQI Iteration {i+1}/{self.n_iterations} complete.")

    def act(self, obs):
        if self.q_model is None or obs[-1] == 0:
            return 0
            
        q0 = self.q_model.predict(self._prepare_features(obs, 0).reshape(1, -1))[0]
        q1 = self.q_model.predict(self._prepare_features(obs, 1).reshape(1, -1))[0]
        
        return 1 if q1 > q0 else 0

### LSPI

In [7]:
class LSPIAgent(BaseAgent):
    def __init__(self, max_iterations=15, tol=1e-3, df=0.95):
        self.max_iterations = max_iterations
        self.tol = tol
        self.df = df
        self.weights = None

    def _get_basis(self, obs, action):
        """
        Constructs basis functions: [1, S, S^2, t, t^2, rights, action, action*S, ...]
        """
        # obs = [S_1/K, ..., S_M/K, t/N, r/L]
        S_norm = obs[:-2]
        t_norm = obs[-2]
        r_norm = obs[-1]
        
        features = [1.0]
        features.extend(S_norm)
        features.extend(S_norm ** 2)
        features.extend([t_norm, t_norm ** 2, r_norm, action])
        
        # Cross terms with action (crucial for linear Q-learning)
        features.extend(S_norm * action)
        features.append(t_norm * action)
        
        return np.array(features)

    def _get_q(self, obs, action):
        if self.weights is None:
            return 0.0
        return self._get_basis(obs, action) @ self.weights

    def act(self, obs):
        if obs[-1] == 0: return 0
        return 1 if self._get_q(obs, 1) > self._get_q(obs, 0) else 0

    def train(self, env, n_episodes):
        transitions = collect_random_transitions(env, n_episodes)
        
        # Determine feature size
        k = len(self._get_basis(transitions[0][0], 0))
        self.weights = np.zeros(k)
        
        for i in range(self.max_iterations):
            A = np.eye(k) * 1e-4  # L2 Regularization to prevent singular matrices
            b = np.zeros(k)
            
            for obs, action, reward, next_obs, done in transitions:
                phi = self._get_basis(obs, action)
                
                if done:
                    phi_next = np.zeros(k)
                else:
                    # Policy Improvement step inside LSTDQ: 
                    # Evaluate the best action in the next state using CURRENT weights
                    next_action = self.act(next_obs)
                    phi_next = self._get_basis(next_obs, next_action)
                
                A += np.outer(phi, (phi - self.df * phi_next))
                b += phi * reward
                
            new_weights = np.linalg.inv(A) @ b
            
            diff = np.linalg.norm(new_weights - self.weights)
            self.weights = new_weights
            print(f"LSPI Iteration {i+1}/{self.max_iterations}, Weight Delta: {diff:.4f}")
            
            if diff < self.tol:
                print("LSPI Converged.")
                break

### Evaluate

In [8]:
def evaluate_policy_forward(test_paths, params, payoff_fn, policy_weights, degree=2):
    """
    Evaluates a learned policy purely sequentially (forward in time) to prevent look-ahead bias.
    """
    n_paths, N_steps, M = test_paths.shape
    N = N_steps - 1
    L = params.L
    delta = params.delta
    df = params.df
    
    cash_flows = np.zeros(n_paths)
    rights_remaining = np.full(n_paths, L)
    cooldown = np.zeros(n_paths) # Tracks steps until next exercise is allowed
    
    for t in range(1, N + 1):
        S_t = test_paths[:, t, :]
        X_t = get_basis_features(S_t, degree)
        payoff_t = payoff_fn(S_t)
        
        can_exercise = (rights_remaining > 0) & (cooldown == 0) & (payoff_t > 0)
        exercise_decision = np.zeros(n_paths, dtype=bool)
        
        # We must check the policy for those who can exercise
        active_idx = np.where(can_exercise)[0]
        
        if len(active_idx) > 0:
            for l in range(1, L + 1):
                # Mask for paths that specifically have 'l' rights remaining
                l_mask = active_idx[rights_remaining[active_idx] == l]
                if len(l_mask) == 0: continue
                
                if t == N:
                    # At expiration, always exercise if ITM
                    exercise_decision[l_mask] = True
                else:
                    weights = policy_weights[t][l]
                    C_t = X_t[l_mask] @ weights['cont']
                    E_future = X_t[l_mask] @ weights['ex'] if l > 1 else 0
                    Total_E_t = payoff_t[l_mask] + E_future
                    
                    exercise_decision[l_mask] = Total_E_t > C_t
        
        # Apply decisions
        exercised_idx = np.where(exercise_decision)[0]
        cash_flows[exercised_idx] += payoff_t[exercised_idx] * (df ** t)
        
        rights_remaining[exercised_idx] -= 1
        cooldown[exercised_idx] = delta + 1 # +1 because we subtract 1 unconditionally below
        
        # Decrement cooldown for all
        cooldown = np.maximum(0, cooldown - 1)
        
    return np.mean(cash_flows)

In [9]:
def evaluate_agent(agent, env, test_paths_indices):
    """
    Evaluates any agent on a specific set of path indices.
    Returns the average discounted reward (the option price lower bound).
    """
    rewards = []
    
    for idx in test_paths_indices:
        obs, _ = env.reset_to_path(idx)
        done = False
        while not done:
            action = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
        rewards.append(info['cumulative_reward'])
        
    return np.mean(rewards), np.std(rewards) / np.sqrt(len(test_paths_indices))

In [10]:
def extract_rl_boundary(agent, params, is_put=True, grid_points=500):
    """
    Extracts the continuous exercise boundary by querying the agent's act() function.
    Works for any algorithm (LSMC, DQN, PPO, FQI) as long as it has an act(obs) method.
    """
    if params.M > 1:
        raise ValueError("2D boundaries are only visualized for M=1")
        
    K, N, L = params.K, params.N, params.L
    spot_grid = np.linspace(1e-4, K * 1.5, grid_points)
    
    boundaries = {l: np.full(N + 1, np.nan) for l in range(1, L + 1)}
    
    for l in range(1, L + 1):
        for t in range(1, N): # t=0 and t=N are edge cases
            r_norm = l / L
            t_norm = t / N
            
            exercise_prices = []
            
            # Query the agent across the grid
            for S in spot_grid:
                # Reconstruct the Gymnasium observation
                obs = np.array([S / K, t_norm, r_norm], dtype=np.float32)
                
                # Check if payoff is strictly positive (ITM)
                is_itm = (K - S) > 0 if is_put else (S - K) > 0
                
                if is_itm and agent.act(obs) == 1:
                    exercise_prices.append(S)
                    
            if exercise_prices:
                # For a Put, the boundary is the highest price where it chose to exercise
                boundaries[l][t] = max(exercise_prices) if is_put else min(exercise_prices)
                
        # Handle Expiration (t = N)
        boundaries[l][N] = K
        
    return boundaries

In [11]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ══════════════════════════════════════════════════════════════════════════════
# 1. HELPERS & WRAPPERS
# ══════════════════════════════════════════════════════════════════════════════

def payoff_put_fn(prices, K):
    """Simple wrapper to match the expected payoff_fn signature for LSM/TD-LSM"""
    if prices.ndim == 1:
        prices = prices.reshape(-1, 1)
    return np.maximum(K - prices.mean(axis=-1), 0.0)

def train_algo(algo, paths, params):
    """
    Trains the algorithm. 
    Returns a dictionary containing the runnable 'model' and any pre-computed 'boundaries'.
    """
    payoff_lambda = lambda S: payoff_put_fn(S, params.K)
    
    if algo == 'LSM':
        # train_lsmc returns (weights, boundaries) directly!
        weights, boundaries = train_lsmc(paths, params, payoff_lambda, is_put=True)
        return {'model': weights, 'boundaries': boundaries}
        
    elif algo == 'TD-LSM':
        weights, boundaries = train_tdlsm(paths, params, payoff_lambda, is_put=True)
        return {'model': weights, 'boundaries': boundaries}
        
    elif algo == 'FQI':
        env = OptionPricingEnv(params, paths)
        agent = FQIAgent(df=params.df)
        agent.train(env, n_episodes=len(paths))
        return {'model': agent, 'boundaries': None} # Boundaries extracted later via act()
        
    elif algo == 'LSPI':
        env = OptionPricingEnv(params, paths)
        agent = LSPIAgent(df=params.df)
        agent.train(env, n_episodes=len(paths))
        return {'model': agent, 'boundaries': None}
        
    else:
        raise ValueError(f"Unknown algorithm: {algo}")

def eval_algo(algo, trained_dict, test_paths, params):
    """Evaluates the model on out-of-sample paths."""
    model = trained_dict['model']
    payoff_lambda = lambda S: payoff_put_fn(S, params.K)
    
    if algo in ('LSM', 'TD-LSM'):
        return evaluate_policy_forward(test_paths, params, payoff_lambda, model)
    elif algo in ('FQI', 'LSPI'):
        env = OptionPricingEnv(params, test_paths)
        mean_reward, _ = evaluate_agent(model, env, range(env.n_paths))
        return mean_reward
    else:
        raise ValueError(f"Unknown algorithm: {algo}")

def get_algo_boundary(algo, trained_dict, params):
    """Routes boundary extraction based on what the algorithm returned/requires."""
    if algo in ('LSM', 'TD-LSM'):
        # Return the pre-calculated boundaries from the training loop
        return trained_dict['boundaries']
    elif algo in ('FQI', 'LSPI'):
        # Query the RL agent's act() function dynamically
        return extract_rl_boundary(trained_dict['model'], params, is_put=True)
    return None

# ══════════════════════════════════════════════════════════════════════════════
# 2. PLOTTING ENGINE
# ══════════════════════════════════════════════════════════════════════════════

ALGO_COLORS = {'LSM': 'gray', 'TD-LSM': 'blue', 'FQI': 'green', 'LSPI': 'purple'}
ALGO_MARKERS = {'LSM': 'o', 'TD-LSM': 's', 'FQI': '^', 'LSPI': 'D'}

def plot_boundaries_all(params, models_map, rights_left, save_path, optimal_bound=None, title_suffix=""):
    plt.figure(figsize=(10, 6), dpi=120)
    fig_color = 'white'
    plt.gcf().set_facecolor(fig_color)
    plt.gca().set_facecolor(fig_color)
    
    time_grid = np.linspace(0, params.T, params.N + 1)
    plt.axhline(params.K, color='black', linestyle='--', linewidth=1.5, label='Strike (K)')
    
    if optimal_bound is not None and rights_left in optimal_bound:
        opt_S = np.full(params.N + 1, np.nan)
        for t, S in optimal_bound[rights_left]:
            if np.isnan(opt_S[t]) or S > opt_S[t]:
                opt_S[t] = S
        opt_S[params.N] = params.K
        plt.plot(time_grid, opt_S, color='red', linewidth=3, label='Exact DP')

    for algo, trained_dict in models_map.items():
        bounds = get_algo_boundary(algo, trained_dict, params)
        if bounds is not None and rights_left in bounds:
            plt.plot(time_grid, bounds[rights_left], 
                     color=ALGO_COLORS.get(algo, 'black'), 
                     linewidth=2, 
                     marker=ALGO_MARKERS.get(algo, 'o'), markersize=4,
                     label=f'{algo}')

    plt.title(f'Exercise Boundary Evolution (Rights={rights_left}) {title_suffix}', fontsize=14, fontweight='bold')
    plt.xlabel('Time (t)', fontsize=12)
    plt.ylabel('Asset Price (S)', fontsize=12)
    plt.xlim(0, params.T)
    plt.ylim(params.K * 0.4, params.K * 1.2)
    plt.legend(loc='lower right', frameon=True, edgecolor='lightgray')
    plt.grid(True, alpha=0.3)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# 3. MAIN EXPERIMENT LOOP
# ══════════════════════════════════════════════════════════════════════════════

def run_experiments(param_configs, algorithms, train_sizes, test_size=10000, n_repeats=3):
    
    for config_idx, config_dict in enumerate(param_configs):
        p = Params(
            S0=config_dict.get('S0', 100.0), K=config_dict.get('K', 100.0), 
            T=config_dict.get('T', 1.0), r=config_dict.get('r', 0.05),
            sigma=config_dict.get('sigma', 0.3), N=config_dict.get('N', 50), 
            M=config_dict.get('M', 1), L=config_dict.get('L', 1),
            delta=config_dict.get('delta', 0)
        )
        process_type = config_dict.get('process', 'gbm')
        compute_optimal = config_dict.get('compute_optimal', False)
        
        print(f"\n{'='*80}")
        print(f"Config {config_idx+1}: N={p.N}, L={p.L}, M={p.M}, σ={p.sigma}, process={process_type}")
        print('='*80)

        folder_name = f"results_N{p.N}_L{p.L}_M{p.M}_{process_type}"
        os.makedirs(folder_name, exist_ok=True)

        test_paths = generate_paths(p, test_size, process=process_type, seed=997)
        
        optimal_price, optimal_bound = None, None
        if compute_optimal and p.M == 1 and p.delta == 0:
            print("Computing Exact DP Baseline...")
            payoff_lambda = lambda S: payoff_put_fn(S, p.K)
            optimal_price, optimal_bound = binomial_tree_swing_1D(p, payoff_lambda)
            print(f"Exact DP Price: {optimal_price:.4f}")

        results = {algo: {'means': [], 'times': []} for algo in algorithms}
        frames_dict = {l: [] for l in range(1, p.L + 1)}

        for n in train_sizes:
            algo_vals  = {a: [] for a in algorithms}
            algo_times = {a: [] for a in algorithms}
            latest_models = {}

            for rep in range(n_repeats):
                train_paths = generate_paths(p, n, process=process_type, seed=42 + rep)

                for algo in algorithms:
                    t0 = time.time()
                    trained_dict = train_algo(algo, train_paths, p)
                    algo_times[algo].append(time.time() - t0)
                    
                    val = eval_algo(algo, trained_dict, test_paths, p)
                    algo_vals[algo].append(val)
                    latest_models[algo] = trained_dict 

            for algo in algorithms:
                results[algo]['means'].append(np.mean(algo_vals[algo]))
                results[algo]['times'].append(np.mean(algo_times[algo]))

            line = f"n={n:>6d}  " + "  ".join(f"{a}={np.mean(algo_vals[a]):.4f}" for a in algorithms)
            print(line)
            
            if p.M == 1:
                for l_rights in range(1, p.L + 1):
                    frame_path = os.path.join(folder_name, f'frame_L{l_rights}_n{n}.png')
                    plot_boundaries_all(
                        params=p, models_map=latest_models, rights_left=l_rights, 
                        save_path=frame_path, optimal_bound=optimal_bound,
                        title_suffix=f"| Train Size: {n}"
                    )
                    frames_dict[l_rights].append(frame_path)

        if p.M == 1:
            for l_rights, frame_paths in frames_dict.items():
                if frame_paths:
                    gif_path = os.path.join(folder_name, f'boundary_evolution_L{l_rights}.gif')
                    imgs = [Image.open(fp) for fp in frame_paths]
                    imgs[0].save(gif_path, save_all=True, append_images=imgs[1:], duration=600, loop=0)
                    for fp in frame_paths: os.remove(fp) 
                    print(f"Saved GIF: {gif_path}")

        # Payoff convergence plot
        plt.figure(figsize=(10, 5), dpi=110)
        if optimal_price:
             plt.axhline(optimal_price, color='red', linestyle='--', linewidth=2, label='Exact DP')
        for algo in algorithms:
            plt.plot(train_sizes, results[algo]['means'], marker=ALGO_MARKERS.get(algo, 'o'),
                     color=ALGO_COLORS.get(algo, 'gray'), label=algo, linewidth=2)
        plt.xscale('log')
        plt.xlabel('Training Paths')
        plt.ylabel('Out-of-Sample Lower Bound Price')
        plt.title(f"Convergence Comparison (N={p.N}, L={p.L})")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(folder_name, 'payoff_convergence.png'))
        plt.close()
        
        # Training time plot
        plt.figure(figsize=(10, 5), dpi=110)
        for algo in algorithms:
            plt.plot(train_sizes, results[algo]['times'], marker=ALGO_MARKERS.get(algo, 's'),
                     color=ALGO_COLORS.get(algo, 'gray'), label=algo, linewidth=2)
        plt.xscale('log')
        plt.yscale('log')
        plt.xlabel('Training Paths')
        plt.ylabel('Training Time (Seconds)')
        plt.title(f"Compute Cost Comparison (N={p.N}, L={p.L})")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(folder_name, 'training_times.png'))
        plt.close()

In [12]:
configs = [
    {'N': 5,  'L': 1,  'delta': 0, 'sigma': 0.7, 'process': 'gbm', 'tag': 'baseline'}
    ]

results = {}
run_experiments(configs, algorithms=['LSM', 'TD-LSM', 'FQI', 'LSPI'], train_sizes=[10], n_repeats=3)



Config 1: N=5, L=1, M=1, σ=0.7, process=gbm
FQI Iteration 1/10 complete.
FQI Iteration 2/10 complete.
FQI Iteration 3/10 complete.
FQI Iteration 4/10 complete.
FQI Iteration 5/10 complete.
FQI Iteration 6/10 complete.
FQI Iteration 7/10 complete.
FQI Iteration 8/10 complete.
FQI Iteration 9/10 complete.
FQI Iteration 10/10 complete.


KeyboardInterrupt: 